# Cav1.2 G406R: backbone geometry and mutant-site proximity

WT and G406R are compared using matched Cα–Cα coordinates in vanilla data. The corrected WT masked candidate is available, but a corrected G406R masked cohort is not; legacy masked data are excluded and the masked mutant comparison is deferred. Mutation-neighborhood Cα panels and a dedicated position-406 panel describe backbone geometry.

Legacy `shortest_*` columns are retained only as historical field names; the current mutant analysis is vanilla-only until corrected G406R masked data exist. WT experimental glycine references occupy a separate panel because they are all-heavy-atom minima and are not an atom-matched mutant comparison. Shorter minimum distances can arise from the added side chain and its rotamers without backbone movement.

## Distance definitions and interpretation

- **Matched backbone metric:** `CA_GLY406_CA-PARTNER_CA` in WT and `CA_ARG406_CA-PARTNER_CA` in G406R. Both endpoints are Cα atoms in both sequences. Missing or nonfinite values stop the new comparison; no shortest-distance fallback is allowed.
- **WT experimental minima:** the minimum Euclidean distance over all non-hydrogen atoms of both residues. Glycine contributes its backbone N, Cα, C and O atoms, not a side chain. The exported reference table identifies the closest atom pair and separately reports Cα–Cα distances.
- **Historical model minima:** the notebooks consume precomputed `shortest_*` CSV columns. Their original PDB-generator atom selection has not been verified. A separate local CIF tool uses glycine Cα and selected terminal atoms, so equivalence to the experimental all-heavy definition must not be assumed.
- **Interpretation:** matched Cα changes measure backbone geometry. Minimum-distance changes combine backbone movement, side-chain reach and rotamers; WT–mutant values additionally change atom composition. Neither proximity nor a 2–4 Å threshold alone establishes hydrogen bonding or a salt bridge. Values below 2 Å are overlap flags.

All snapshot panels use the explicitly selected cohort below and are descriptive, not tests treating snapshots as independent replicates. WT markers in the Cα panels use matched Cα endpoints; WT minima in the legacy section are contextual references on a separate axis.

In [1]:
from pathlib import Path
import importlib
import sys
repo_root = Path.cwd() if (Path.cwd() / 'shared').is_dir() else Path.cwd().parent
sys.path.insert(0, str(repo_root)) if str(repo_root) not in sys.path else None
import shared.mutation_site_analysis as mutation_site_analysis
importlib.reload(mutation_site_analysis)  # Pick up helper changes in an already-running kernel.
load_ensemble = mutation_site_analysis.load_ensemble
nearby_distance_columns = mutation_site_analysis.nearby_distance_columns
rank_nearby_shifts = mutation_site_analysis.rank_nearby_shifts
top_aliases = mutation_site_analysis.top_aliases
plot_nearby_overlay = mutation_site_analysis.plot_nearby_overlay
from shared.dataset_selection import distance_csv_options, load_selected_distance_csv, cav12_active_masked_distance_path
from shared.plotting import CAV12_PALETTE, plot_top_shifts
G406R_EXPERIMENTAL_DISTANCES = {
    '8WE6 | chain A': {
        'CA_LEU401_CA-ILE1186_CA': [10.070], 'CA_LEU401_CA-VAL753_CA': [11.718],
        'CA_GLU407_CA-ARG1532_CA': [9.716], 'CA_GLU407_CA-LEU1530_CA': [7.270], 'CA_GLU407_CA-THR1531_CA': [6.724],
        'CA_PHE408_CA-ARG1532_CA': [12.668], 'CA_PHE408_CA-THR1531_CA': [9.273], 'GLU407-ARG1532': [8.011],
        'GLU407-ASP1533': [9.304], 'GLU407-TYR1529': [8.236], 'LEU404-ARG1532': [11.220],
        'PHE408-ARG1532': [10.269], 'PHE408-TRP1534': [13.153], 'SER405-ARG1532': [9.744],
    },
    '8HLP | chain A': {
        'CA_LEU401_CA-ILE1186_CA': [9.986], 'CA_LEU401_CA-VAL753_CA': [11.751],
        'CA_GLU407_CA-ARG1532_CA': [8.687], 'CA_GLU407_CA-LEU1530_CA': [7.352], 'CA_GLU407_CA-THR1531_CA': [6.392],
        'CA_PHE408_CA-ARG1532_CA': [11.595], 'CA_PHE408_CA-THR1531_CA': [8.828], 'GLU407-ARG1532': [7.179],
        'GLU407-ASP1533': [8.362], 'GLU407-TYR1529': [7.913], 'LEU404-ARG1532': [10.248],
        'PHE408-ARG1532': [9.330], 'PHE408-TRP1534': [12.814], 'SER405-ARG1532': [9.122],
    },
    '8FD7 | chain K': {
        'CA_LEU401_CA-ILE1186_CA': [13.653], 'CA_LEU401_CA-VAL753_CA': [15.182],
        'CA_GLU407_CA-ARG1532_CA': [11.739], 'CA_GLU407_CA-LEU1530_CA': [10.872], 'CA_GLU407_CA-THR1531_CA': [8.756],
        'CA_PHE408_CA-ARG1532_CA': [8.953], 'CA_PHE408_CA-THR1531_CA': [6.103], 'GLU407-ARG1532': [9.949],
        'GLU407-ASP1533': [10.982], 'GLU407-TYR1529': [11.953], 'LEU404-ARG1532': [8.867],
        'PHE408-ARG1532': [5.747], 'PHE408-TRP1534': [10.327], 'SER405-ARG1532': [6.412],
    },
}
G406R_EXPERIMENTAL_DISTANCES

from shared.plotting import apply_kv21_style
apply_kv21_style()


In [2]:
from shared.presentation_export import install_notebook_figure_export
figure_dir = repo_root / 'cav12' / 'dataDistances' / 'analysis' / 'figures'
table_dir = repo_root / 'cav12' / 'dataDistances' / 'analysis' / 'tables'
figure_dir.mkdir(parents=True, exist_ok=True)
table_dir.mkdir(parents=True, exist_ok=True)
# Export every displayed panel locally, including the neighborhood comparisons.
install_notebook_figure_export(figure_dir, 'Cav12_G406R_mutationSite_analysis')


PosixPath('/Users/ahernandezgonzalez/Repositories/vgci_mutants/cav12/dataDistances/analysis/figures/Cav12_G406R_mutationSite_analysis')

In [3]:
DATASET_SELECTION = 'all_ok_3'  # Corrected 3 Å convergence subset generated from the complete 2026-07-25 G406R tables.
data = repo_root / 'cav12' / 'dataDistances'
DATASET_PATHS = {
    'WT vanilla': distance_csv_options(repo_root, data / '26-02-10_Cav12_wt_vanillaAF2_distances_all.csv', 'Cav12', 'WT', 'vanilla'),
    'WT masked': distance_csv_options(repo_root, cav12_active_masked_distance_path(repo_root, 'WT'), 'Cav12', 'WT', 'masked'),
    'G406R vanilla': distance_csv_options(repo_root, data / '26-07-25_Cav1.2_g406r_vanillaAF2_distances_all.csv', 'Cav12', 'G406R', 'vanilla'),
    'G406R masked': distance_csv_options(repo_root, cav12_active_masked_distance_path(repo_root, 'G406R'), 'Cav12', 'G406R', 'masked'),
}
DATASET_PATHS
wt_van = load_selected_distance_csv('WT vanilla', DATASET_PATHS['WT vanilla'], DATASET_SELECTION, fallback_to_all=False)
wt_mask = load_selected_distance_csv('WT masked', DATASET_PATHS['WT masked'], DATASET_SELECTION, fallback_to_all=False)
mut_van = load_selected_distance_csv('G406R vanilla', DATASET_PATHS['G406R vanilla'], DATASET_SELECTION, fallback_to_all=False)
mut_mask = load_selected_distance_csv('G406R masked', DATASET_PATHS['G406R masked'], DATASET_SELECTION, fallback_to_all=False)
structures = {'8WE6 | chain A': (repo_root / 'cav12' / 'experimental' / '8WE6.pdb', 'A'), '8HLP | chain A': (repo_root / 'cav12' / 'experimental' / '8HLP.pdb', 'A'), '8FD7 | chain K': (repo_root / 'cav12' / 'experimental' / '8FD7.pdb', 'K')}
experimental_colors = None  # shared PDB-specific colors and markers


/Users/ahernandezgonzalez/miniforge3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


WT vanilla: requested=all_ok_3; actual=all_ok_3; path=/Users/ahernandezgonzalez/Repositories/vgci_mutants/cav12/dataDistances/26-02-10_Cav12_wt_vanillaAF2_distances_all_ok_rmsd_3A.csv
Merged companion distance table: cav12/dataDistances/26-11-10_Cav12_wt_masked_v3_50pct_AF2_distances_extra_all_ok_rmsd_3A.csv
WT masked: requested=all_ok_3; actual=all_ok_3; path=cav12/dataDistances/26-11-10_Cav12_wt_masked_v3_50pct_AF2_distances_all_ok_rmsd_3A.csv


G406R vanilla: requested=all_ok_3; actual=all_ok_3; path=/Users/ahernandezgonzalez/Repositories/vgci_mutants/cav12/dataDistances/26-07-25_Cav1.2_g406r_vanillaAF2_distances_all_ok_rmsd_3A.csv
Merged companion distance table: cav12/dataDistances/26-11-10_Cav12_g406r_masked_v3_50pct_AF2_distances_extra_all_ok_rmsd_3A.csv
G406R masked: requested=all_ok_3; actual=all_ok_3; path=cav12/dataDistances/26-11-10_Cav12_g406r_masked_v3_50pct_AF2_distances_all_ok_rmsd_3A.csv


## Shared mutation-neighborhood Cα coordinates

These ranked comparisons include only Cα–Cα columns present in both sequences. They describe surrounding backbone geometry; the explicit mutation-site comparison follows below. Historical shortest-distance columns are excluded from this backbone ranking.

In [4]:
for protocol, wt, mutant, colors in [('vanilla', wt_van, mut_van, (CAV12_PALETTE['WT_VAN'], CAV12_PALETTE['G406R_VAN'])), ('masked', wt_mask, mut_mask, (CAV12_PALETTE['WT_HM'], CAV12_PALETTE['G406R_HM']))]:
    shifts = rank_nearby_shifts(wt, mutant, [c for c in nearby_distance_columns(wt, 406) if c.startswith("CA_")])
    display(shifts.head(20))
    if shifts.empty:
        print(f'No complete WT–G406R paired distances are available for {protocol}; skipping the ranked-shift overlay.')
        continue
    if shifts.empty:
        print(f'Skipping {protocol}: no mutation-site distance columns contain values in both WT and G406R ensembles.')
        continue
    plot_top_shifts(shifts, f'Largest Cav1.2 mutation-neighborhood Cα shifts: G406R {protocol}')
    experimental_aliases = set(next(iter(G406R_EXPERIMENTAL_DISTANCES.values())))
    shifts = shifts[shifts['distance'].map(lambda column: column.replace('CA_CA_', '').replace('shortest_', '') in experimental_aliases)]
    if shifts.empty:
        print(f'Skipping {protocol}: none of the comparable columns has an explicit experimental distance.')
        continue
    aliases = top_aliases(shifts, n=len(experimental_aliases))
    experimentals = {structure: {alias: values for alias, values in distances.items() if alias in aliases} for structure, distances in G406R_EXPERIMENTAL_DISTANCES.items()}
    plot_nearby_overlay(wt, mutant, aliases, 'Cav1.2', 'G406R', protocol, colors, experimentals, experimental_colors)



,distance,wt_median_A,mutant_median_A,median_shift_A,abs_shift_A
81,CA_GLU407_CA-ARG1532_CA,8.92,10.28,1.36,1.36
80,CA_GLU407_CA-THR1531_CA,9.26,10.54,1.28,1.28
98,CA_PHE408_CA-ARG1532_CA,11.93,13.19,1.26,1.26
97,CA_PHE408_CA-THR1531_CA,12.65,13.89,1.24,1.24
79,CA_GLU407_CA-LEU1530_CA,12.79,14.01,1.22,1.22
96,CA_PHE408_CA-LEU1530_CA,15.91,17.03,1.12,1.12
83,CA_GLU407_CA-TRP1534_CA,12.62,13.68,1.06,1.06
100,CA_PHE408_CA-TRP1534_CA,16.05,17.05,1.00,1.00
78,CA_GLU407_CA-TYR1529_CA,12.68,13.67,0.99,0.99
46,CA_LEU404_CA-THR1531_CA,11.11,12.02,0.91,0.91


/Users/ahernandezgonzalez/Repositories/vgci_mutants/shared/presentation_export.py:51: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  return original_show(*args, **kwargs)


/Users/ahernandezgonzalez/Repositories/vgci_mutants/shared/presentation_export.py:51: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  return original_show(*args, **kwargs)


,distance,wt_median_A,mutant_median_A,median_shift_A,abs_shift_A
64,CA_SER405_CA-ARG1532_CA,11.280,13.265,1.985,1.985
98,CA_PHE408_CA-ARG1532_CA,11.985,13.830,1.845,1.845
63,CA_SER405_CA-THR1531_CA,10.720,12.560,1.840,1.840
60,CA_SER405_CA-ASP1528_CA,8.740,10.550,1.810,1.810
62,CA_SER405_CA-LEU1530_CA,13.310,15.100,1.790,1.790
61,CA_SER405_CA-TYR1529_CA,12.550,14.330,1.780,1.780
97,CA_PHE408_CA-THR1531_CA,12.100,13.850,1.750,1.750
80,CA_GLU407_CA-THR1531_CA,8.810,10.510,1.700,1.700
81,CA_GLU407_CA-ARG1532_CA,9.460,11.090,1.630,1.630
96,CA_PHE408_CA-LEU1530_CA,15.550,17.180,1.630,1.630


/Users/ahernandezgonzalez/Repositories/vgci_mutants/shared/presentation_export.py:51: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  return original_show(*args, **kwargs)


/Users/ahernandezgonzalez/Repositories/vgci_mutants/shared/presentation_export.py:51: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  return original_show(*args, **kwargs)


## Matched position-406 Cα comparison

The same partner residues appear in the backbone and legacy-proximity figures. WT glycine and mutant ARG406 contribute Cα only in this panel; experimental markers also use Cα–Cα distances. The exported table records each source column, model identifier, and selected cohort.

In [5]:
from shared.cav12_glycine_analysis import plot_matched_ca
from shared.mutation_contact_analysis import cav12_experimental_shortest_distances
import matplotlib.pyplot as plt

partners = ['THR1531', 'ASP1533', 'ASP1528', 'PHE1527']
experimental_geometry = cav12_experimental_shortest_distances(repo_root, 406, partners)
comparisons = {'vanilla': (wt_van, mut_van), 'masked': (wt_mask, mut_mask)}
matched_ca, matched_ca_figure = plot_matched_ca(
    comparisons, 406, 'ARG406', 'G406R', partners, experimental_geometry
)
matched_ca.to_csv(table_dir / 'g406r_matched_ca_plot_data.csv', index=False)
display(matched_ca.groupby(['Protocol', 'Sequence', 'Partner'])['Cα distance (Å)'].agg(['count', 'median']))
for extension in ('png', 'pdf'):
    matched_ca_figure.savefig(figure_dir / f'g406r_matched_ca_comparison.{extension}', dpi=300, bbox_inches='tight')
plt.show()


count  median
Protocol Sequence Partner               
masked   G406R    ASP1528   4998   8.045
                  ASP1533   4998  11.190
                  PHE1527   4998   9.680
                  THR1531   4998   9.440
         WT       ASP1528   4992   6.430
                  ASP1533   4992  10.230
                  PHE1527   4992   8.370
                  THR1531   4992   7.270
vanilla  G406R    ASP1528   5000   7.130
                  ASP1533   5000  10.460
                  PHE1527   5000   9.620
                  THR1531   5000   9.060
         WT       ASP1528   5000   6.330
                  ASP1533   5000  10.190
                  PHE1527   5000   9.100
                  THR1531   5000   7.880

/Users/ahernandezgonzalez/Repositories/vgci_mutants/shared/presentation_export.py:51: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  return original_show(*args, **kwargs)


## R406-centered overlap screen and conditional proximity

A snapshot **passed the R406-centered overlap screen** when none of the selected R406-centered legacy distances was below 2 Å. Passing establishes only that these selected checks did not trigger the overlap rule; it does not validate the entire pocket.

The pass fraction is presented alongside the D1528 and D1533 proximity summaries because those contact fractions are conditional on passing. The pass fraction fell from approximately 67.4% in vanilla to 21.9% after masking.

In [6]:
from shared.cav12_glycine_analysis import plot_g406r_screen_and_contacts
r406_screen_summary, r406_screen_figure = plot_g406r_screen_and_contacts(repo_root)
r406_screen_summary.to_csv(table_dir / 'g406r_screen_and_conditional_contacts.csv', index=False)
display(r406_screen_summary)
for extension in ('png', 'pdf'):
    r406_screen_figure.savefig(figure_dir / f'g406r_screen_and_conditional_contacts.{extension}', dpi=300, bbox_inches='tight')
plt.show()


,protocol,overlap_threshold_A,total_snapshots,passing_snapshots,unconditional_snapshot_pass_fraction,total_nominal_trajectories,trajectories_with_any_passing_snapshot,trajectory_any_pass_fraction,snapshot_pass_denominator_definition,trajectory_pass_denominator_definition,...,conditional_contact_denominator_passing_snapshots,conditional_contact_fraction,unconditional_contact_denominator_all_snapshots,unconditional_pass_and_contact_fraction,contact_rule,partner_residue,atom_selection_rule,unconditional_pass_fraction,conditional_contact_denominator,conditional_contact_numerator
0,vanilla,2.0,5000,3371,0.674200,500,475,0.950,all retained QC-qualified snapshots,all retained nominal seed x AF2-model trajecto...,...,NaN,NaN,NaN,NaN,NaN,screen_only,side_chain_non_hydrogen_atoms,0.674200,NaN,NaN
1,vanilla,2.0,5000,3371,0.674200,500,475,0.950,all retained QC-qualified snapshots,all retained nominal seed x AF2-model trajecto...,...,3371.0,0.101454,5000.0,0.068400,distance <= 4.0 A,ASP1528,side_chain_non_hydrogen_atoms,0.674200,3371.0,342.0
2,vanilla,2.0,5000,3371,0.674200,500,475,0.950,all retained QC-qualified snapshots,all retained nominal seed x AF2-model trajecto...,...,3371.0,0.422723,5000.0,0.285000,distance <= 4.0 A,ASP1533,side_chain_non_hydrogen_atoms,0.674200,3371.0,1425.0
3,masked,2.0,4998,576,0.115246,500,224,0.448,all retained QC-qualified snapshots,all retained nominal seed x AF2-model trajecto...,...,NaN,NaN,NaN,NaN,NaN,screen_only,side_chain_non_hydrogen_atoms,0.115246,NaN,NaN
4,masked,2.0,4998,576,0.115246,500,224,0.448,all retained QC-qualified snapshots,all retained nominal seed x AF2-model trajecto...,...,576.0,0.112847,4998.0,0.013005,distance <= 4.0 A,ASP1528,side_chain_non_hydrogen_atoms,0.115246,576.0,65.0
5,masked,2.0,4998,576,0.115246,500,224,0.448,all retained QC-qualified snapshots,all retained nominal seed x AF2-model trajecto...,...,576.0,0.135417,4998.0,0.015606,distance <= 4.0 A,ASP1533,side_chain_non_hydrogen_atoms,0.115246,576.0,78.0


/Users/ahernandezgonzalez/Repositories/vgci_mutants/shared/presentation_export.py:51: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  return original_show(*args, **kwargs)


## Mutant proximity and separate WT glycine context

The frequency bars and split violins compare **mutant vanilla with mutant masked**, not WT with mutant. Their values come from legacy model `shortest_*` columns; atom-selection provenance remains unresolved. The 2–4 Å and <2 Å bins are proximity and overlap summaries, not validated interactions.

The separate WT experimental panel uses all-heavy-atom minima, including glycine backbone atoms. It must not be subtracted from the mutant minima to infer mutation-induced backbone movement. The matched Cα panels above provide the backbone comparison.

In [7]:
import pandas as pd

# Historical exported-distance sensitivity
# These values are retained to reconcile the pooled-snapshot and hierarchical
# estimands. The historical closest-atom identities cannot be recovered.
historical_reconciliation = pd.read_csv(
    repo_root / 'analysis' / 'statistics_revision' / 'paired_seed_v2'
    / 'g406r_conditional_contact_estimand_reconciliation.csv'
)
display_columns = [
    'partner', 'protocol', 'pooled_passing_snapshots',
    'pooled_contact_numerator', 'pooled_contact_denominator',
    'pooled_conditional_fraction', 'conditional_trajectories',
    'per_trajectory_conditional_estimate',
    'seed_balanced_conditional_estimate', 'whole_seed_CI_low',
    'whole_seed_CI_high',
]
display(historical_reconciliation[display_columns].round(4))
historical_reconciliation.to_csv(
    table_dir / 'g406r_historical_estimand_reconciliation.csv', index=False
)


,partner,protocol,pooled_passing_snapshots,pooled_contact_numerator,pooled_contact_denominator,pooled_conditional_fraction,conditional_trajectories,per_trajectory_conditional_estimate,seed_balanced_conditional_estimate,whole_seed_CI_low,whole_seed_CI_high
0,ASP1528,vanilla,3371.0,342.0,3371.0,0.1015,475.0,0.2156,0.2162,0.2070,0.2240
1,ASP1528,masked,576.0,65.0,576.0,0.1128,224.0,0.1475,0.1404,0.1004,0.1856
2,ASP1528,masked_minus_vanilla,NaN,NaN,NaN,0.0114,NaN,-0.0681,-0.0759,-0.1163,-0.0280
3,ASP1533,vanilla,3371.0,1425.0,3371.0,0.4227,475.0,0.3276,0.3307,0.3117,0.3508
4,ASP1533,masked,576.0,78.0,576.0,0.1354,224.0,0.1803,0.1561,0.1208,0.1926
5,ASP1533,masked_minus_vanilla,NaN,NaN,NaN,-0.2873,NaN,-0.1473,-0.1746,-0.2144,-0.1350
